In [1]:
from bertopic import BERTopic
import pandas as pd
from collections import defaultdict
import numpy as np
from collections import Counter

/Users/lcarv/PycharmProjects/risklive/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def make_topic_keyword_column(df):
    df['RelevantKeywords_new'] = df['RelevantKeywords'].str.split(', ')
    grouped = df.groupby('topic')['RelevantKeywords_new'].sum().reset_index()
    def get_top_3_keywords(keywords_list):
        all_keywords = [keyword for keyword in keywords_list]
        keyword_counts = Counter(all_keywords)
        top_3 = ', '.join([kw for kw, _ in keyword_counts.most_common(3)])
        return top_3
    grouped['topic_keyword'] = grouped['RelevantKeywords_new'].apply(get_top_3_keywords)
    df = df.merge(grouped[['topic', 'topic_keyword']], on='topic', how='left')
    return df    

In [4]:
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.express as px
from plotly.subplots import make_subplots
import webbrowser

def map_nuclear(row):
    if row['NewsCategory'] == "nuclear industry":
        return "nuclear"
    else:
        return row['NewsCategory']


def create_hyperlink(url):
    return f'<a href="{url}" style="cursor: pointer" target="_blank" rel="noopener noreferrer">🔗</a>'


def create_two_treemaps(data):
    data = data[data.topic != -1]
    data = make_topic_keyword_column(data)
    data = data[data['AlertFlag'].isin(['Red', 'Yellow'])]  # Filter out Green alerts
    data['NewsCategory'] = data.apply(map_nuclear, axis=1)
    data['URL'] = data['URL'].apply(create_hyperlink)
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    
    alert_levels = ['Red', 'Yellow']
    dataframes = {level: data[data['AlertFlag'] == level] for level in alert_levels}
    
    fig = go.FigureWidget(make_subplots(rows=2, cols=1, 
                        row_heights=[0.5, 0.5],
                        specs=[[{'type': 'treemap'}], [{'type': 'treemap'}]],
                        subplot_titles=("High Risk", "Medium Risk"),
                        vertical_spacing=0.05))
    
    # High Risk treemap
    high_risk_treemap = px.treemap(
        dataframes['Red'],
        path=['AlertFlag', 'NewsCategory', 'topic_keyword', 'RelevantKeywords', 'URL', 'Title'],
        color='AlertFlag',
        color_discrete_map={'Red': 'red'},
        custom_data=['URL']
    )
    
    high_risk_treemap.update_traces(
        hovertemplate='<span style="font-size: 20px;"><b>%{label}</b><br>Count: %{value}<br>',
        marker=dict(cornerradius=5),
        textfont=dict(size=25)
    )
    
    for trace in high_risk_treemap.data:
        fig.add_trace(trace, row=1, col=1)
    
    # Medium Risk treemap
    medium_risk_treemap = px.treemap(
        dataframes['Yellow'],
        path=['AlertFlag', 'NewsCategory', 'topic_keyword', 'RelevantKeywords', 'URL', 'Title'],
        color='AlertFlag',
        color_discrete_map={'Yellow': 'yellow'},
        custom_data=['URL']
    )
    
    medium_risk_treemap.update_traces(
        hovertemplate='<span style="font-size: 20px;"><b>%{label}</b><br>Count: %{value}<br>',
        marker=dict(cornerradius=5),
        textfont=dict(size=25)
    )
    
    for trace in medium_risk_treemap.data:
        fig.add_trace(trace, row=2, col=1)

    def on_click(trace, points, state):
        if points.point_inds:
            ind = points.point_inds[0]
            url = trace.customdata[ind][0]
            if url and url != 'nan':
                webbrowser.open_new_tab(url)

    for trace in fig.data:
        trace.on_click(on_click)
    
    fig.update_layout(
        height=800,
        margin=dict(t=80, l=25, r=25, b=25),
        title={
            'text': '<b>News Article Risk Assessment</b><br><sup>Categorized by Alert Level, News Category, Topic, and Keywords</sup>',
            'y': 0.98,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 24, 'color': 'black'}
        },
        clickmode='event+select'
    )
    
    for annotation in fig.layout.annotations:
        annotation.font.update(size=14)    
    return fig

In [28]:
df = pd.read_csv("/home/azureuser/github/risk_live/results/data/df_with_response_and_topics.csv")
fig = create_two_treemaps(df)

FileNotFoundError: [Errno 2] No such file or directory: '/home/azureuser/github/risk_live/results/data/df_with_response_and_topics.csv'

In [21]:
fig

NameError: name 'fig' is not defined

In [22]:
def convert_datetime_to_ymd(dt):
    ymd = dt.strftime("%Y-%m-%d")
    return ymd
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='ISO8601').dt.tz_convert('UTC')
df['timestamps_new'] = df['Timestamp'].apply(convert_datetime_to_ymd)
df['timestamps_new'].value_counts() 

NameError: name 'df' is not defined

In [23]:
df[df['timestamps_new']=="2024-09-07"]

,Title,URL,Description,Timestamp,IsTrending,LLM_Response,LLM_Price,LLM_Token_Usage,PromptTokens,CompletionTokens,...,RelevantKeywords,ShortSummary,Relevance,RelevanceReason,AlertFlag,AlertReason,NewsCategory,API_Timestamp,topic,timestamps_new
0,Two counter-protesters detained as pro-Palesti...,https://www.msn.com/en-gb/news/world/two-count...,Two counter-protesters detained as pro-Palesti...,2024-09-07 13:41:00+00:00,no,"{'RelevantKeywords': ['counter-protesters', 'p...",0.006678,"CompletionUsage(completion_tokens=143, prompt_...",1244.0,143.0,...,"counter-protesters, pro-Palestine, march, deta...",Two counter-protesters were detained as pro-Pa...,No,NaN,Green,The article does not pertain to the Nuclear De...,miscellaneous,2024-09-07 14:00:06,24,2024-09-07
1,"Despite climate extremes, Bangladesh improves ...",https://www.iaea.org/bulletin/despite-climate-...,Bangladesh — a country highly vulnerable to fl...,2024-09-07 13:56:00+00:00,no,"{'RelevantKeywords': ['Bangladesh', 'climate c...",0.006765,"CompletionUsage(completion_tokens=149, prompt_...",1248.0,149.0,...,"Bangladesh, climate crisis, agricultural secto...","Bangladesh, facing climate extremes such as fl...",No,NaN,Green,The article discusses agricultural challenges ...,miscellaneous,2024-09-07 14:00:14,19,2024-09-07
2,The UAE Has Completed the Arab World's First N...,https://www.uaemoments.com/the-uae-has-complet...,The Barakah Nuclear Energy Plant in Abu Dhabi ...,2024-09-07 13:45:00+00:00,no,"{'RelevantKeywords': ['UAE', 'Barakah Nuclear ...",0.007165,"CompletionUsage(completion_tokens=193, prompt_...",1217.0,193.0,...,"UAE, Barakah Nuclear Energy Plant, nuclear, el...",The UAE has completed the Barakah Nuclear Ener...,Yes,The completion of the Barakah Nuclear Energy P...,Yellow,The news is of moderate importance as it highl...,geopolitical,2024-09-07 14:00:16,47,2024-09-07
3,6 countries with nuclear submarines,https://timesofindia.indiatimes.com/etimes/tre...,The world was first introduced to nuclear subm...,2024-09-07 13:30:00+00:00,no,"{'RelevantKeywords': ['nuclear submarines', 'C...",0.006554,"CompletionUsage(completion_tokens=138, prompt_...",1228.0,138.0,...,"nuclear submarines, Cold War, global defense, ...",The article discusses the introduction of nucl...,No,NaN,Green,The article provides historical context on nuc...,miscellaneous,2024-09-07 14:00:18,11,2024-09-07
4,How nuclear and climate-smart agriculture solu...,https://www.iaea.org/bulletin/how-nuclear-and-...,Climate-smart agriculture involves monitoring ...,2024-09-07 13:39:00+00:00,no,"{'RelevantKeywords': ['nuclear', 'climate-smar...",0.006999,"CompletionUsage(completion_tokens=174, prompt_...",1232.0,174.0,...,"nuclear, climate-smart agriculture, climate ch...",The article discusses how nuclear and climate-...,No,The article focuses on the intersection of nuc...,Green,The content is not directly relevant to the ND...,miscellaneous,2024-09-07 14:00:25,-1,2024-09-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,Workers ‘cheated’ out of holiday pay – TUC,https://www.msn.com/en-gb/money/careersandeduc...,TUC - More than a million employees did not ge...,2024-09-07 23:32:00+00:00,no,"{'RelevantKeywords': ['holiday pay', 'TUC', 'e...",0.006213,"CompletionUsage(completion_tokens=112, prompt_...",1220.0,112.0,...,"holiday pay, TUC, employees",A TUC study suggests that more than a million ...,No,NaN,Green,The article discusses issues related to employ...,miscellaneous,2024-09-08 08:01:23,20,2024-09-07
230,Vaughan Gething to stand down from Senedd at n...,https://www.msn.com/en-gb/news/uknews/vaughan-...,Vaughan Gething to stand down from Senedd at n...,2024-09-07 20:05:00+00:00,no,"{'RelevantKeywords': ['Vaughan Gething', 'Sene...",0.006531,"CompletionUsage(completion_tokens=133, prompt_...",1237.0,133.0,...,"Vaughan Gething, Senedd, Labour, election","Vaughan Gething, a Labour former first ministe...",No,NaN,Green,The news is about a political figure's decisio.

In [22]:
df[df['timestamps_new']=="2024-08-13"]

,Title,URL,Description,Timestamp,IsTrending,LLM_Response,LLM_Price,LLM_Token_Usage,PromptTokens,CompletionTokens,...,RelevantKeywords,ShortSummary,Relevance,RelevanceReason,AlertFlag,AlertReason,NewsCategory,API_Timestamp,topic,timestamps_new
1501,Why astronauts on an eight-day mission could b...,https://www.msn.com/en-gb/news/world/why-astro...,Most of us have experienced the frustration of...,2024-08-13 17:18:00+00:00,no,"{'RelevantKeywords': ['astronauts', 'mission',...",0.006396,"CompletionUsage(completion_tokens=121, prompt_...",1239.0,121.0,...,"astronauts, mission, space",Astronauts on an eight-day mission could be st...,No,NaN,Green,The article discusses issues related to space ...,miscellaneous,2024-09-10 14:00:23,-1,2024-08-13


In [21]:
df[df['timestamps_new']=="2024-09-05"]

,Title,URL,Description,Timestamp,IsTrending,LLM_Response,LLM_Price,LLM_Token_Usage,PromptTokens,CompletionTokens,...,RelevantKeywords,ShortSummary,Relevance,RelevanceReason,AlertFlag,AlertReason,NewsCategory,API_Timestamp,topic,timestamps_new
1500,New Mexico attorney general sues company behin...,https://www.msn.com/en-gb/news/world/new-mexic...,New Mexico attorney general sues company behin...,2024-09-05 19:54:00+00:00,no,"{'RelevantKeywords': ['New Mexico', 'attorney ...",0.006761,"CompletionUsage(completion_tokens=154, prompt_...",1232.0,154.0,...,"New Mexico, attorney general, Snapchat, child ...",The New Mexico attorney general has filed a la...,No,The article does not pertain to the Nuclear De...,Green,The content is not relevant to the NDA's inter...,miscellaneous,2024-09-10 14:00:22,-1,2024-09-05


In [12]:
from datetime import datetime, timedelta
import pytz

df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='ISO8601').dt.tz_convert('UTC')
cutoff_date = datetime.now(pytz.UTC) - timedelta(days=3)
# df_new = df[df['Timestamp'] >= cutoff_date]
df_new = df
def convert_datetime_to_ymd(dt):
    ymd = dt.strftime("%Y-%m-%d")
    return ymd
df_new['timestamps_new'] = df_new['Timestamp'].apply(convert_datetime_to_ymd)
df_new['timestamps_new']

0       2024-09-07
1       2024-09-07
2       2024-09-07
3       2024-09-07
4       2024-09-07
           ...    
1482    2024-09-10
1483    2024-09-10
1484    2024-09-10
1485    2024-09-10
1486    2024-09-10
Name: timestamps_new, Length: 1487, dtype: object

In [31]:
from risklive.server import clean_old_data
clean_old_data()

165

In [30]:
import os
from risklive.config import SAVE_DIR
csv_dir = SAVE_DIR['CSV_DATA_DIR']
os.listdir(csv_dir)

['df_with_response_and_topics.csv',
 'news_data_with_llm_info.csv',
 'news_data.csv']

In [7]:
topics = topic_model.get_topics()

def find_overlapping_words(topics):
    overlaps = {}
    for topic1 in topics:
        for topic2 in topics:
            if topic1 != topic2 and topic1!=-1:
                words1 = set(word for word, _ in topics[topic1])
                words2 = set(word for word, _ in topics[topic2])
                overlap = words1.intersection(words2)
                if overlap:
                    overlaps[(topic1, topic2)] = overlap
    return overlaps

overlapping_words = find_overlapping_words(topics)

NameError: name 'topic_model' is not defined

In [8]:
def get_topic_words(topic_model, topic_id, top_n=10):
    return [word for word, _ in topic_model.get_topic(topic_id)][:top_n]

def find_topics_to_merge(topic_model, similarity_threshold=3):
    topics = topic_model.get_topics()
    topic_words = {topic: set(get_topic_words(topic_model, topic)) for topic in topics}
    topics_to_merge = []
    merged_topics = set()

    for topic1 in topics:
        if topic1 in merged_topics:
            continue
        for topic2 in topics:
            if topic1 < topic2 and topic2 not in merged_topics and topic1!=-1:
                common_words = topic_words[topic1].intersection(topic_words[topic2])
                if len(common_words) >= similarity_threshold:
                    topics_to_merge.append([topic1, topic2])
                    merged_topics.add(topic1)
                    merged_topics.add(topic2)
                    break  # Move to the next topic1

    return topics_to_merge

In [ ]:
topics_to_merge = find_topics_to_merge(topic_model, similarity_threshold=2)
        

In [ ]:
topics_to_merge

In [9]:

def get_topic_words(topic_model, topic_id, top_n=10):
    return [word for word, _ in topic_model.get_topic(topic_id)][:top_n]

def find_similar_topics(topic_model, similarity_threshold=3):
    topics = topic_model.get_topics()
    topic_words = {topic: set(get_topic_words(topic_model, topic)) for topic in topics}
    similar_topics = defaultdict(list)

    for topic1 in topics:
        for topic2 in topics:
            if topic1 < topic2:  # Avoid duplicate comparisons
                common_words = topic_words[topic1].intersection(topic_words[topic2])
                if len(common_words) >= similarity_threshold:
                    similar_topics[topic1].append((topic2, common_words))

    return similar_topics

def iterative_topic_merging(topic_model, docs, similarity_threshold=3):
    while True:
        similar_topics = find_similar_topics(topic_model, similarity_threshold)
        if not similar_topics:
            break

        # Merge the first pair of similar topics found
        topic1 = next(iter(similar_topics))
        topic2, _ = similar_topics[topic1][0]
        topics_to_merge = [topic1, topic2]

        # Use BERTopic'query built-in merge_topics method
        topic_model.merge_topics(docs, topics_to_merge)

        print(f"Merged topics {topic1} and {topic2}")

    return topic_model

## TREEMAP

In [11]:
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

def map_nuclear(row):
    if row['NewsCategory']=="nuclear industry":
        return "nuclear"
    else:
        return row['NewsCategory']
    
# def create_hyperlink(url):
#     return f'<a href="{url}" target="_blank">🔗</a>'

def create_hyperlink(url):
    return f'<a xlink:href="{url}" style="cursor: pointer" target="_blank" rel="noopener noreferrer">News Link</a>'


def create_three_treemaps(data):
    data['NewsCategory'] = data.apply(map_nuclear, axis=1)
    data['URL'] = data['URL'].apply(create_hyperlink)
    # Ensure data is a DataFrame
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    
    # Create separate dataframes for each alert level
    alert_levels = ['Red', 'Yellow', 'Green']
    dataframes = {level: data[data['AlertFlag'] == level] for level in alert_levels}
    
    # Create a subplot with 1 row and 3 columns
    fig = make_subplots(rows=1, cols=3, 
                        column_widths=[0.5, 0.30, 0.20], 
                        specs=[[{'type': 'treemap'}, {'type': 'treemap'}, {'type': 'treemap'}]],
                        subplot_titles=("High Risk", "Medium Risk", "Low Risk"),
                        horizontal_spacing=0.01)
    
    # Create and add each treemap to the subplot
    for i, (level, df) in enumerate(dataframes.items(), start=1):
        treemap = px.treemap(
            df,
            path=['AlertFlag', 'NewsCategory', 'topic', 'RelevantKeywords', 'URL'],
            color='AlertFlag',
            color_discrete_map={'Red': 'red', 'Yellow': 'yellow', 'Green': 'green'},
            custom_data=['URL']
        )
        
        treemap.update_traces(
            hovertemplate='<b>%{label}</b><br>Count: %{value}',
            marker=dict(cornerradius=5),
            textfont=dict(size=15)  # Increase the font size here
        )
        
        # Add the treemap to the main figure
        for trace in treemap.data:
            fig.add_trace(trace, row=1, col=i)
    
    # Update the layout of the main figure
    fig.update_layout(
        height=600,
        margin=dict(t=80, l=25, r=25, b=25),
        title={
            'text': '<b>News Article Risk Assessment</b><br><sup>Categorized by Alert Level, News Category, Topic, and Keywords</sup>',
            'y': 0.95,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 24, 'color': 'black'}
        }
    )
    
    # Remove individual treemap titles
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=14)
    
    return fig

In [12]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import webbrowser

def map_nuclear(row):
    if row['NewsCategory'] == "nuclear industry":
        return "nuclear"
    else:
        return row['NewsCategory']

def create_hyperlink(url):
    return f'<a href="{url}" style="cursor: pointer" target="_blank" rel="noopener noreferrer">🔗 News Link</a>'

def create_three_treemaps(data):
    data['NewsCategory'] = data.apply(map_nuclear, axis=1)
    data['URL_display'] = data['URL'].apply(create_hyperlink)
    
    # Ensure data is a DataFrame
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    
    # Create separate dataframes for each alert level
    alert_levels = ['Red', 'Yellow', 'Green']
    dataframes = {level: data[data['AlertFlag'] == level] for level in alert_levels}
    
    # Create a FigureWidget with 1 row and 3 columns
    fig = go.FigureWidget(make_subplots(rows=1, cols=3, 
                        column_widths=[0.5, 0.30, 0.20], 
                        specs=[[{'type': 'treemap'}, {'type': 'treemap'}, {'type': 'treemap'}]],
                        subplot_titles=("High Risk", "Medium Risk", "Low Risk"),
                        horizontal_spacing=0.01))
    
    # Create and add each treemap to the subplot
    for i, (level, df) in enumerate(dataframes.items(), start=1):
        treemap = px.treemap(
            df,
            path=['AlertFlag', 'NewsCategory', 'topic', 'RelevantKeywords', 'URL_display'],
            color='AlertFlag',
            color_discrete_map={'Red': 'red', 'Yellow': 'yellow', 'Green': 'green'},
            custom_data=['URL']
        )
        
        treemap.update_traces(
            hovertemplate='<b>%{label}</b><br>Count: %{value}<br>',
            marker=dict(cornerradius=5),
            textfont=dict(size=15)
        )
        
        # Add the treemap to the main figure
        for trace in treemap.data:
            fig.add_trace(trace, row=1, col=i)

    def on_click(trace, points, state):
        if points.point_inds:
            ind = points.point_inds[0]
            url = trace.customdata[ind]
            if url and url != 'nan':  # Check if URL is not empty or NaN
                webbrowser.open_new_tab(url)

    for trace in fig.data:
        trace.on_click(on_click)
    
    # Update the layout of the main figure
    fig.update_layout(
        height=600,
        margin=dict(t=80, l=25, r=25, b=25),
        title={
            'text': '<b>News Article Risk Assessment</b><br><sup>Categorized by Alert Level, News Category, Topic, and Keywords</sup>',
            'y': 0.95,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 24, 'color': 'black'}
        },
        clickmode='event+select'
    )
    
    # Update subplot titles
    for annotation in fig.layout.annotations:
        annotation.font.update(size=14)
    
    return fig

In [13]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import webbrowser

def map_nuclear(row):
    if row['NewsCategory'] == "nuclear industry":
        return "nuclear"
    else:
        return row['NewsCategory']


def create_hyperlink(url):
    return f'<a href="{url}" style="cursor: pointer" target="_blank" rel="noopener noreferrer">🔗</a>'


def create_three_treemaps(data):
    data['NewsCategory'] = data.apply(map_nuclear, axis=1)
    data['URL'] = data['URL'].apply(create_hyperlink)
    # Ensure data is a DataFrame
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    
    # Create separate dataframes for each alert level
    alert_levels = ['Red', 'Yellow', 'Green']
    dataframes = {level: data[data['AlertFlag'] == level] for level in alert_levels}
    
    # Create a FigureWidget with 1 row and 3 columns
    fig = go.FigureWidget(make_subplots(rows=1, cols=3, 
                        column_widths=[0.5, 0.30, 0.20], 
                        specs=[[{'type': 'treemap'}, {'type': 'treemap'}, {'type': 'treemap'}]],
                        subplot_titles=("High Risk", "Medium Risk", "Low Risk"),
                        horizontal_spacing=0.01))
    
    # Create and add each treemap to the subplot
    for i, (level, df) in enumerate(dataframes.items(), start=1):
        treemap = px.treemap(
            df,
            path=['AlertFlag', 'NewsCategory', 'RelevantKeywords'],
            color='AlertFlag',
            color_discrete_map={'Red': 'red', 'Yellow': 'yellow', 'Green': 'green'},
            custom_data=['URL']
        )
        
        treemap.update_traces(
            # hovertemplate='<b>%{label}</b><br>Count: %{value}<br>',
            hovertemplate='<span style="font-size: 20px;"><b>%{label}</b><br>Count: %{value}<br>',
            marker=dict(cornerradius=5),
            textfont=dict(size=15)
        )
        
        # Add the treemap to the main figure
        for trace in treemap.data:
            fig.add_trace(trace, row=1, col=i)

    # def on_click(trace, points, state):
    #     if points.point_inds:
    #         ind = points.point_inds[0]
    #         url = trace.customdata[ind][0]
    #         if url and url != 'nan':  # Check if URL is not empty or NaN
    #             webbrowser.open_new_tab(url)

    # for trace in fig.data:
    #     trace.on_click(on_click)
    
    # Update the layout of the main figure
    fig.update_layout(
        height=600,
        margin=dict(t=80, l=25, r=25, b=25),
        title={
            'text': '<b>News Article Risk Assessment</b><br><sup>Categorized by Alert Level, News Category, Topic, and Keywords</sup>',
            'y': 0.95,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 24, 'color': 'black'}
        },
        clickmode='event+select'
    )
    
    # Update subplot titles
    for annotation in fig.layout.annotations:
        annotation.font.update(size=14)
    
    return fig


In [14]:
import pandas as pd
import plotly.express as px
import numpy as np 
df = pd.read_csv("../results/data/df_with_response_and_topics.csv")
df = df[df.topic!=-1]
treemap = create_three_treemaps(df)
treemap.show()

FileNotFoundError: [Errno 2] No such file or directory: '../results/data/df_with_response_and_topics.csv'

In [15]:
import os
import pickle
with open(os.path.join("treemap.pkl"), 'wb') as f:
    pickle.dump(treemap, f)

NameError: name 'treemap' is not defined

In [16]:
path = "/home/azureuser/github/risk_live/notebooks/treemap.pkl"
with open(path, 'rb') as f:
    fig2 = pickle.load(f)
fig2.show()

FileNotFoundError: [Errno 2] No such file or directory: '/home/azureuser/github/risk_live/notebooks/treemap.pkl'